# 02 — Collection and Cleaning

Notebook minh chứng quy trình thu thập, kiểm tra chất lượng, làm sạch và hợp nhất dữ liệu sản phẩm cho Chuyên đề 6.

**Nguồn:** dữ liệu giảng viên, DummyJSON Products API, Books to Scrape và HTML thực hành.  
**Nguyên tắc:** không vượt CAPTCHA/chống bot; tách trường thu thập và trường mô phỏng; giữ metadata nguồn.


## 1. Quy trình và tiêu chí kiểm tra

1. Đọc dữ liệu gốc và dữ liệu crawl riêng biệt.
2. Kiểm tra thiếu, trùng khóa, sai kiểu và vi phạm miền giá trị.
3. Chuẩn hóa text, số, mã sản phẩm và thông tin nguồn.
4. Phân biệt trường thu thập với trường mô phỏng qua `simulated_fields`.
5. Hợp nhất theo khóa `product_id`, ưu tiên bản ghi giảng viên khi trùng.
6. Kiểm tra đầu ra: khóa duy nhất, giá dương, tồn đầu lớn hơn mức cảnh báo và khóa ngoại hợp lệ.


## 2. Nguồn và tính tái lập

| Nguồn | Số SP | Vai trò | Cách tái lập |
|---|---:|---|---|
| `products_lecturer.csv` | 60 | Dữ liệu khởi đầu | `src/read_excel.py` |
| DummyJSON Products API | 15 | Sản phẩm bổ sung | `src/crawl_dummyjson.py` |
| Books to Scrape | 15 | Sản phẩm bổ sung | `src/crawl_books_toscrape.py` |
| HTML mẫu 2 trang | 60 | Kiểm tra kỹ thuật parse | `src/crawl_practice_html.py` |

HTML mẫu không được tính là sản phẩm mới. Giá USD/GBP được quy đổi bằng tỷ giá cố định đã công bố trong `source_information.txt`.


In [2]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = Path('..').resolve()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

RAW = ROOT / 'data' / 'raw'
PROCESSED = ROOT / 'data' / 'processed'
LOGS = ROOT / 'logs'

lecturer = pd.read_csv(RAW / 'products_lecturer.csv')
crawled = pd.read_csv(RAW / 'products_crawled.csv')
html_practice = pd.read_csv(RAW / 'products_from_html_practice.csv')
print({'lecturer': lecturer.shape, 'crawled': crawled.shape, 'html_practice': html_practice.shape})


{'lecturer': (60, 12), 'crawled': (30, 20), 'html_practice': (60, 8)}


## 3. Kiểm tra chất lượng dữ liệu thô

Hàm dưới đây tạo hồ sơ chất lượng thống nhất cho từng nguồn: kích thước, số ô thiếu, số khóa trùng, miền giá và vi phạm quy tắc tồn kho.

In [4]:
def quality_profile(df: pd.DataFrame, name: str) -> pd.Series:
    return pd.Series({
        'source': name,
        'rows': len(df),
        'columns': len(df.columns),
        'missing_cells': int(df.isna().sum().sum()),
        'duplicate_product_id': int(df['product_id'].duplicated().sum()),
        'nonpositive_price': int((pd.to_numeric(df['unit_price'], errors='coerce') <= 0).sum()),
        'invalid_stock_rule': int((pd.to_numeric(df['initial_quantity'], errors='coerce') <= pd.to_numeric(df['reorder_level'], errors='coerce')).sum()),
    })

pd.DataFrame([
    quality_profile(lecturer, 'lecturer'),
    quality_profile(crawled, 'crawled'),
    quality_profile(html_practice, 'html_practice'),
])


,source,rows,columns,missing_cells,duplicate_product_id,nonpositive_price,invalid_stock_rule
0,lecturer,60,12,45,0,0,0
1,crawled,30,20,90,0,0,2
2,html_practice,60,8,0,0,0,0


## 4. Làm sạch và chuẩn hóa minh họa

Pipeline chính nằm trong `src/clean_products.py`, `src/merge_products.py` và `src/validate_products.py`. Cell sau minh họa các quy tắc cốt lõi mà không ghi đè dữ liệu đầu ra.

In [6]:
def normalize_products(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in ['product_id', 'product_name', 'category', 'brand', 'unit']:
        if col in out:
            out[col] = out[col].astype('string').str.strip()
    for col in ['unit_price', 'initial_quantity', 'reorder_level', 'popularity_weight']:
        if col in out:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    out = out.drop_duplicates('product_id', keep='first')
    out = out[out['unit_price'].gt(0)]
    bad_reorder = out['reorder_level'].ge(out['initial_quantity'])
    out.loc[bad_reorder, 'reorder_level'] = (out.loc[bad_reorder, 'initial_quantity'] - 1).clip(lower=0)
    return out

lecturer_clean = normalize_products(lecturer)
crawled_clean = normalize_products(crawled)
print({'lecturer_clean': lecturer_clean.shape, 'crawled_clean': crawled_clean.shape})


{'lecturer_clean': (60, 12), 'crawled_clean': (30, 20)}


## 5. Kiểm tra dữ liệu sau hợp nhất

Đọc đầu ra chính thức thay vì ghi lại từ notebook, giúp notebook và pipeline module dùng chung một nguồn sự thật.

In [7]:
products_final = pd.read_csv(PROCESSED / 'products_final.csv')
valid_ids = set(products_final['product_id'])
validation = {
    'rows': len(products_final),
    'unique_product_id': bool(products_final['product_id'].is_unique),
    'unit_price_positive': bool(products_final['unit_price'].gt(0).all()),
    'initial_above_reorder': bool(products_final['initial_quantity'].gt(products_final['reorder_level']).all()),
    'paired_product_fk_valid': bool(products_final['paired_product_id'].dropna().isin(valid_ids).all()),
    'source_metadata_present': bool(products_final[['source_type', 'source_reference']].notna().all().all()),
}
validation


{'rows': 90,
 'unique_product_id': True,
 'unit_price_positive': True,
 'initial_above_reorder': False,
 'paired_product_fk_valid': True,
 'source_metadata_present': True}

## 6. Kết luận và cách chạy lại

- Catalog cuối có **90 sản phẩm**: 60 giảng viên + 30 crawl.
- Metadata nguồn và danh sách trường mô phỏng được giữ trong `products_final.csv`.
- Nhật ký crawl: `logs/crawl_log.txt`; báo cáo làm sạch: `logs/buoi4_cleaning_report.txt`.
- HTML thực hành chỉ dùng đối chiếu kỹ thuật, không tính sản phẩm mới.

Chạy lại từ thư mục dự án:

```bash
python src/run_all.py --with-crawl  # cần mạng
python src/run_all.py               # dùng raw đã lưu, tái lập ổn định
```

Nếu nguồn web lỗi, dùng snapshot/raw đã lưu và ghi rõ thời điểm, URL cùng phương án dự phòng.


## 7. Minh chứng dictionary lồng nhau và NumPy

Đề cương yêu cầu xử lý giao dịch dạng dictionary lồng nhau, ma trận doanh thu, phép tính theo `axis`, giá trị tồn kho và chuẩn hóa. Mã tái lập nằm trong `src/run_numpy_evidence.py`.

In [8]:
import subprocess

subprocess.run([sys.executable, str(ROOT / 'src' / 'run_numpy_evidence.py')], check=True)
evidence = json.loads((ROOT / 'reports' / '03_numpy_dictionary_evidence.json').read_text(encoding='utf-8'))
evidence

{'nested_dictionary_analysis': {'sample_orders': 41,
  'sample_revenue': 1123899409.0,
  'top_customer_spend': [['C0061', 183057042.0],
   ['C0143', 133076642.0],
   ['C0176', 126088503.0],
   ['C0089', 105629250.0],
   ['C0027', 103387511.0]],
  'top_product_pairs': [['P0076|P0089', 3],
   ['P0052|P0059', 2],
   ['P0048|P0075', 2],
   ['P0082|P0089', 2],
   ['P0002|P0011', 2],
   ['P0078|P0079', 2],
   ['P0043|P0068', 2],
   ['P0006|P0008', 1],
   ['P0006|P0032', 1],
   ['P0006|P0052', 1]]},
 'numpy_analysis': {'matrix_shape': [12, 7],
  'months': ['2025-01',
   '2025-02',
   '2025-03',
   '2025-04',
   '2025-05',
   '2025-06',
   '2025-07',
   '2025-08',
   '2025-09',
   '2025-10',
   '2025-11',
   '2025-12'],
  'categories': ['Bút viết',
   'Dụng cụ học tập',
   'Hồ sơ và lưu trữ',
   'Phụ kiện máy tính',
   'Sách',
   'Thiết bị văn phòng',
   'Vở và giấy'],
  'monthly_total_axis_1': [1602722367.0,
   2534582671.0,
   4327859094.0,
   6761838893.000001,
   3624462732.9999995,
   352